# 对比与判读 — 各次训练到底哪个更好

训练在 [`MSN_train_skullfix.ipynb`](MSN_train_skullfix.ipynb)，**这里只负责判读**。

| 节 | 回答什么问题 |
|---|---|
| 1. 选 run | 要比哪几个 |
| **2. 指标词典** | 每个数字是什么意思、**主指标是哪个**、什么时候该盯哪个 |
| **3. 三把尺子 + 判决清单** | "这个差异是真的吗"——三种噪声口径分别回答什么 |
| 4. 训练曲线 | 收敛正不正常、学习率降了几次 |
| **5. 同轮次表** | 去掉"跑得久天然占便宜"这个混淆 |
| 6. 逐颅骨评估 | 主表：全点云 + 缺损区（**占显存**） |
| **7. 配对检验** | 换一批颅骨还成不成立 |
| 8. 分布图 | 均值会骗人，看重叠程度 |
| 9. 可视化 / 10. 对照组 | 形状对不对、和预训练权重比 |

> ⚠️ 第 6 节之后本 kernel 会占住显存。**回去训练之前必须 Restart Kernel**，否则训练子进程 OOM。

## 1. 选 run

`RUNS` 里每加一行，下面所有表和图就多一列。缺权重的会自动跳过
（要么还没跑，要么已按有效性分界裁剪）。

⛔ **不要把 `baseline_es20` / `dcd_w3` / `dcd_l2` / `rep05_void` 加回来**：它们跑在
学习率衰减从未触发的配置错误下，"最优值"是随机游走上的最小值，与任何其它轮的差值
都不可归因。权重也已删除。

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd

REPO = os.path.abspath(".")
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, "src", "models")):
    REPO = os.path.dirname(REPO)
sys.path.insert(0, os.path.join(REPO, "src", "eval"))
sys.path.insert(0, os.path.join(REPO, "src", "models"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("HF_HOME", "/root/.cache/huggingface")

import importlib, report as rp
importlib.reload(rp)      # 改过 report.py 之后必须 reload，否则拿到的是缓存的旧模块

# ============================ 要比哪几个 ============================
RUNS = [
    ("cd_only",       "msn_skullfix/cd_only"),        # 2x2: CD 单独
    ("lr_fix",        "msn_skullfix/lr_fix_only"),    # 2x2: CD+DCD
    ("rep_w05",       "msn_skullfix/rep_w05"),        # 2x2: CD+DCD+rep
    ("cd_rep05_full", "msn_skullfix/cd_rep05_full"),  # 2x2: CD+rep  ← 目前的最优配置
    ("cd_rep05_r2",   "msn_skullfix/cd_rep05_r2"),    # 同配置重复 → 这是噪声底线的来源
    ("tie_qk",        "msn_skullfix/tie_qk"),         # Q/K 初始绑定
    ("tie_qk_r2",     "msn_skullfix/tie_qk_r2"),      # 它的重复实验（跑完自动出现）
    ("notext",        "msn_skullfix/notext"),         # 去掉文本分支
    ("pp_attn",       "msn_skullfix/pp_attn"),        # 逐点交叉注意力 ❌ 已否决
]
BASE = "cd_rep05_full"     # 第 7 节配对检验的比较基准
# ===================================================================

_missing = [(n, p) for n, p in RUNS
            if not os.path.exists(os.path.join(REPO, "experiments", p, "best.h5"))]
if _missing:
    print("无权重，已跳过（未跑，或已按有效性分界裁剪）:",
          ", ".join(n for n, _ in _missing), "\n")
runs = rp.load_runs(REPO, [r for r in RUNS if r not in _missing])

print(f"{'run':16}{'配置':34}{'轮数':>6}{'停止':>16}{'LR降':>6}{'CD_t(run.json)':>16}")
for r in runs:
    n_ep, best_ep = r.meta["epochs_run"], int(r.hist["val_loss"].idxmin()) + 1
    # 先判早停：它是决定性的（最后 patience 轮没有改善）。反过来先判上限会误伤 ——
    # 早期的 run.json 没记 --epochs，回退值可能比它实际用的上限小。
    if n_ep - best_ep == r.meta["early_stop_patience"]:
        stop = "EarlyStopping"
    elif "epochs" not in r.meta:
        stop = "⚠️ 上限未记录"
    elif n_ep >= r.meta["epochs"]:
        stop = "❌ 被上限截断"
    else:
        stop = "⚠️ 墙钟掐停"
    print(f"{r.label:16}{r.config_str():34}{n_ep:>6}{stop:>16}"
          f"{len(r.lr_drops):>6}{r.meta['best_val_cd_t_mm']:>16.3f}")

assert len({tuple(r.meta["val_ids"]) for r in runs}) == 1, "各 run 的验证集划分不同，不可比"
print(f"\n所有 run 共用同一批 {len(runs[0].meta['val_ids'])} 颗验证颅骨 ✅")
print("架构:", ", ".join(sorted({r.arch_label for r in runs})))

## 2. 指标词典

### 2.1 主指标是 `defect_cov_mm`（缺损区覆盖）

**只有约 6.7% 的 GT 点落在缺损区**，其余 93.3% 是输入里已经给了、模型只需复现的表面。
所以全点云指标主要在衡量"抄得像不像"——一个把可见区域复现得完美、却把洞填成一团糟的
模型，在全点云表里依然好看。

实测也支持这个选择：2×2 消融在**缺损覆盖上四格全部统计显著**，而同样四格在**全点云 CD_t
上全部不显著**（置信区间都跨零）。不是偏好问题，是全点云指标分辨不出这些改动。

### 2.2 缺损区（主表）

| 列 | 是什么 | 能被糊弄吗 |
|---|---|---|
| **`defect_cov_mm`** ⭐ | 缺损区每个 GT 点到最近预测点的平均距离 = **洞填得全不全** | ✅ **不能**。不填洞这个数直接爆掉（把输入原样当预测：12.94mm） |
| `defect_HD95_mm` | 同上的 95 分位 = 洞里最差的那块 | ✅ 不能 |
| `defect_prec_mm` | 放进洞里的预测点，离真实表面多远 = **填得准不准** | ⚠️ **能**——一个点都不放就是 nan。**必须和 `defect_n_pred` 一起看**。而且实测**在损失函数这一维度上不区分模型**（各配置 2.89~3.01mm），换初始化才动得了 |
| `defect_n_pred` | 放进洞里的预测点数 | GT 约 395 个，各配置放 387~492 个 = 数量基本正确 |
| `defect_gt_%` | 缺损区占 GT 的比例 | 这是**数据的属性不是模型的**，各 run 完全相同，不用比 |

**"算不算在缺损区"两侧规则不同，这是刻意的**（2026-08-20 修正过一次）：

- **GT 侧**：到最近的**输入点** > 5mm（"这块表面输入里没有"）
- **预测侧**：到最近的**缺损 GT 点** < 5mm（"这个点落在洞里"）

早先两侧用同一条规则，结果把洞周围一圈漂移点也圈了进来——那实际在量"预测点飘离表面多远"
（CD_t/HD95 已经覆盖了），并且**对越差的模型圈得越多**，把 `defect_prec_mm` 虚高了 43%。
阈值 5mm 取自 GT→输入距离双峰分布的**谷底**（峰在 2-3mm 和 15mm+）。

### 2.3 全点云（辅表）

| 列 | 是什么 | 什么时候用 |
|---|---|---|
| `CD_t_mm` | 双向平均最近邻距离之和 | 通用精度。**别单独用它下结论**——93.3% 的权重在复现输入上 |
| `HD95_mm` | 95 分位最近邻距离 = **最坏情况** | 临床角度。CD 是均值，会把"某处差 15mm"平均掉，而那正是不可接受的 |
| `F1@0.05` / `F1@0.03` | 阈值内点占比的调和平均（≈5.19mm / 3.11mm） | **能和原论文 Table 1/2 直接并排的两个之一** |
| `DCD` | 密度感知 Chamfer，统一按 λ=1 报告 | 另一个可跨项目比的：论文报 1.41269，用他们权重实测 1.42772（差 1%）→ 复现正确性的证据 |
| `clump_%` | 最近邻 < 2mm 的点占比（**GT = 0.0%**） | **密度问题的直接读数**，repulsion 就是冲它去的 |
| `spacing_CV` | 点间距的变异系数（**GT = 0.145**） | 同上，但更平滑、噪声更小 |

**不报 `CD_p`**：它的定义是 `sqrt(平均距离)`——对长度开根号，量纲不成立（从原项目原样继承的 bug）。

### 2.4 什么时候盯哪个 ⭐

| 你改的是 | 主要盯 | 理由 |
|---|---|---|
| 损失函数 / 密度项（repulsion、DCD） | `clump_%` + `spacing_CV`，再看 `defect_cov_mm` | 密度效应极大（20/20 颅骨一致），最灵敏 |
| 架构 / 初始化（tie_qk、pp_attn、no-text） | `defect_cov_mm`，`CD_t_mm` 辅 | 这类改动的效应只有 0.02~0.09mm，全点云上分辨不出 |
| 想说"**补全能力**变强了" | 只有 `defect_cov_mm` 算数 | 唯一不能被糊弄的那个 |
| 想和**原论文**并排 | `DCD`、`F1@0.05/0.03` | ⚠️ `CD` 不能跨项目比（他们的 0.00170 比点间距还小 35 倍，量纲对不上） |
| 想说"离**临床**多远" | `HD95` + Poisson 重建对照（还没做） | 点云 HD95 地板 3.79mm，而 AutoImplant 冠军 1.52mm（0.45mm 体素网格）——**差距主要来自表示方式** |

### 2.5 采样地板 —— 这条决定了还有多少空间

同一张网格独立采样两次（模型误差为零）实测：**CD_t 地板 4.43mm、HD95 地板 3.79mm**，
单向约 **2.2mm**。`defect_cov_mm` 是单向指标，所以当前的 3.24mm 里约 2.2mm 是采样分辨率，
**模型自身只贡献约 1mm** → 这条线的优化空间比看上去小得多。

⚠️ **本文数值不可与体素域方法直接并排**，这一条必须写进论文。

## 3. 三把尺子 + 判决清单

### 3.1 三种"噪声"，回答三个不同的问题

| 尺子 | 怎么量 | 回答 | 当前值 |
|---|---|---|---|
| ① **同配置重跑** | 同一配置训两次比均值 | "重训一次数字会不会变" = 训练随机性 | CD_t **0.004mm**、缺损覆盖 **0.0065mm**。⚠️ 只有**一对**样本、**一个**配置，且**随轮数变**（同一对截到 150 轮差 0.108mm，退火后才 0.007mm）——**它不是万能底线** |
| ② **逐颅骨配对检验** | 第 7 节 | "换一批颅骨还成不成立" = 可推广性 | 颅骨间 CD_t std 约 1.0mm，配对后 SE 约 0.06mm |
| ③ **末段 epoch 抖动** | 末 30 轮 std | "报告的最优值本身抖多少" | 退火后 0.003~0.010mm；没退火时 ~0.25mm（前四轮作废的原因） |

**三者互不替代。** 此前把 ① 当唯一尺子，于是把一个 10/20 的抛硬币（`tie_qk` 的缺损覆盖）
描述成了"10.6× 噪声"。

而**训练长度**（222~411 轮）根本不是噪声，是**系统性混淆**——见第 5 节。

### 3.2 一个改动算"成立"，要同时满足

- [ ] **同轮次表上仍然领先**（第 5 节）——不是靠多跑几十轮换来的
- [ ] **配对检验在主指标上过关**（第 7 节）：`p_wilcoxon < 0.002`，或改善颅骨数 ≥ 17/20
      （做了 ~24 个检验，0.05 那条线不够用）
- [ ] **有一次同配置重复**，且两次的差异 < 你声称的效应
- [ ] **方向和机制解释一致**（不能只有"跑出来是这样"）

**不满足就写进 devlog 标 ⚠️ 待确认，不进论文。** 项目里已经有两条栽在这上面
（`--no-text` n=1；`tie_qk` 归因不干净）。

## 4. 训练曲线

虚线是学习率下降。**没有虚线的 run 直接作废**——那说明衰减从未触发。

In [ ]:
rp.fig_curves(runs).show()

## 5. 同轮次表 ⭐

各 run 由 EarlyStopping 自己停，停在 222~411 轮不等，而报告的是**全程最优的那一轮**。
所以"碰巧还在改善"的 run 天然占便宜——**这不是噪声，是系统性混淆**。

实测：`tie_qk` 领先 0.095mm，但它跑了 411 轮；两边都截到 249 轮，领先只剩 **0.02~0.03mm**，
即**三分之二的优势来自多跑的 160 轮**。

**怎么读**：
- 在最短的那个共同轮次上仍然领先 → 是配置的功劳
- 只在它自己的停止点领先 → 你量到的是"这次训练跑得久"。这**也可能**是配置的真实性质
  （比如某个初始化让模型收敛更慢但最终解更好），但那是**另一句话**，需要重复实验才能分开。

In [ ]:
at = sorted({222, 249, 300, min(len(r.hist) for r in runs)})
print(rp.epoch_matched(runs, at=at).to_string())
print("\n列的含义: reported = 全程最优（各 run 自己的停止点）；@N = 都截到第 N 轮时的最优；"
      "\nlate_std = 末 30 轮抖动（尺子③）。单位 mm，history 口径。")

## 6. 逐颅骨评估 —— 主表

**这一节开始占显存**（每套架构建一个 187M 模型）。回去训练前记得 Restart Kernel。

推理是**完全确定性的**（验证时走固定种子的 stateless 采样），所以只要权重还在，
这张表随时能逐位重算——这一点和训练不同，训练在 GPU 上不可复现。

In [ ]:
eval_df = rp.eval_runs(REPO, runs)     # 每颗验证颅骨一行
print()
print("=== 缺损区（主表）===")
print(rp.format_defect_summary(eval_df))
print()
print("=== 全点云（辅表）===")
print(rp.format_summary(eval_df))

### 6.1 存档（可选）

`experiments_log/eval_all_runs.csv` 是**跟踪进 git 的冻结记录**，它现在还含有
`baseline` / `dcd_l2` 两行——那两个 run 的权重已删除、**再也算不出来**。

所以这里是**按 run 名合并**，不是覆盖。直接覆盖会把那两行永久删掉。

In [ ]:
SAVE = False        # 确认表没问题之后改成 True

if SAVE:
    path = os.path.join(REPO, "experiments_log", "eval_all_runs.csv")
    merged = eval_df
    if os.path.exists(path):
        prev = pd.read_csv(path)
        merged = pd.concat([prev[~prev["run"].isin(eval_df["run"])], eval_df], ignore_index=True)
    merged.to_csv(path, index=False)
    print(f"-> {path}  ({merged['run'].nunique()} 个 run / {len(merged)} 行)")
else:
    print("SAVE=False，没有写盘。确认上面的表没问题后改成 True 再跑一次。")

## 7. 配对检验 ⭐ —— "换一批颅骨还成不成立"

所有 run 评估的是**同一批 20 颗**颅骨，所以可以逐颗配对，把"这颗颅骨本身就难"消掉。
这很重要：颅骨之间的差异（CD_t std ≈ 1.0mm）比要找的效应（0.02~0.2mm）大一个数量级，
不配对的话全都会被淹没。

**怎么读**：
- `delta` = 改动后 − 基准；`↓好` 的指标里负数是改善
- `改善 18/20` 这一列往往比 `delta` 更有说服力——均值的置信区间常常很宽，
  但"20 颗里 18 颗都变好"本身就是强证据
- `p_sign` = 改善的颅骨数多于随机吗；`p_wilcoxon` = 连幅度也一致吗
- ⚠️ **做了很多个检验，用 p < 0.002 这条线**，不是 0.05

⚠️ 它**看不到训练随机性**（两个模型是固定的）。所以配对检验过关 ≠ 结论成立，
还需要尺子①（重复实验）。

In [ ]:
for r in runs:
    if r.label == BASE:
        continue
    print(rp.format_paired(eval_df, BASE, r.label,
                           cols=["defect_cov_mm", "defect_HD95_mm", "defect_prec_mm",
                                 "CD_t_mm", "HD95_mm", "F1@0.05", "clump_%", "spacing_CV"]))
    print()

## 8. 分布图

均值会骗人。箱线图的**重叠程度**才是"这个差异是否可信"的直接证据——
两个箱子几乎完全重叠时，那 0.05mm 的均值差别不要当成改进。

In [ ]:
rp.fig_per_skull(eval_df, "defect_cov_mm").show()    # 主指标
rp.fig_per_skull(eval_df, "CD_t_mm").show()
rp.fig_progress(eval_df, baseline=BASE).show()       # 所有指标翻转成「向下 = 变好」

## 9. 可视化（点云）

蓝色是预测的完整颅骨，浅红是缺损输入——**缺损区应该只有蓝色**。

> 散点图只能看"形状对不对"。要看表面质量和点的疏密，用
> [`MSN_surface_quality.ipynb`](MSN_surface_quality.ipynb)：那边有 mesh 重建、
> 有符号偏差着色和间距诊断图。

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import tensorflow as tf
import msn_skullfix as msn

SHOW_RUN = BASE          # 改这里换 run

_run = next(r for r in runs if r.label == SHOW_RUN)
for _g in tf.config.experimental.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_g, True)

# 架构由这个 run 自己的 run.json 决定，不要写死 paper()。改解码器 key 的来源或关掉
# 文本分支都会换一套拓扑，而前者不改变任何权重形状 —— 旧 checkpoint 会被静默读进
# 新拓扑、不报错，然后给出一份属于「从未训练过的网络」的图。
_cfg = rp.arch_config(msn, _run.arch_key)
model = msn.build_model(_cfg)
model.load_weights(_run.weights)

_data = np.load(os.path.join(REPO, "data", "cache", "skullfix_pairs_4096_6144.npz"))
ids, inputs, gt = _data["ids"], _data["inputs"], _data["gt"]
_val = _run.meta["val_ids"]
val_pos = [int(np.where(ids == v)[0][0]) for v in _val]
_x = [inputs[val_pos]]
if _cfg.use_text:
    _x.append(np.tile(np.load(os.path.join(REPO, "data", "cache", "bert_skull.npy"))[None],
                      (len(val_pos), 1)))
# predict 而不是 model(x) 逐个调用：后者实测每次泄漏 0.29 GiB 且不释放
preds = model.predict(_x, batch_size=1, verbose=0)
print(f"{SHOW_RUN} ({_run.arch_label}): {model.count_params()/1e6:.1f}M 参数, {len(preds)} 颗")

PRED, INP, GTC = rp.C_TRAIN, "#EF9A9A", rp.C_GT


def show_completion(k, camera=(1.6, 1.6, 1.2)):
    pred, pos, sid = preds[k], val_pos[k], _val[k]
    fig = go.Figure([
        go.Scatter3d(x=pred[:, 0], y=pred[:, 1], z=pred[:, 2], mode="markers",
                     name="Predicted complete skull",
                     marker=dict(size=1.6, color=PRED, opacity=0.85)),
        go.Scatter3d(x=inputs[pos][:, 0], y=inputs[pos][:, 1], z=inputs[pos][:, 2],
                     mode="markers", name="Defective input",
                     marker=dict(size=1.5, color=INP, opacity=0.45))])
    fig.update_layout(title=f"skull_{sid} ({SHOW_RUN}) — 缺损区应只有蓝色", height=650,
                      legend=dict(itemsizing="constant", x=0.02, y=0.98,
                                  bgcolor="rgba(255,255,255,0.6)"),
                      scene=dict(aspectmode="data", camera=dict(eye=dict(zip("xyz", camera)))),
                      margin=dict(l=0, r=0, b=0, t=40))
    return fig


def show_pred_vs_gt(k):
    pred, pos, sid = preds[k], val_pos[k], _val[k]
    fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scatter3d"}] * 2],
                        subplot_titles=(f"Prediction ({SHOW_RUN})", "Ground truth"))
    fig.add_trace(go.Scatter3d(x=pred[:, 0], y=pred[:, 1], z=pred[:, 2], mode="markers",
                               name="Predicted", marker=dict(size=1.4, color=PRED)), 1, 1)
    g = gt[pos]
    fig.add_trace(go.Scatter3d(x=g[:, 0], y=g[:, 1], z=g[:, 2], mode="markers",
                               name="Ground truth", marker=dict(size=1.4, color=GTC)), 1, 2)
    fig.update_layout(height=520, title=f"skull_{sid}",
                      legend=dict(itemsizing="constant", orientation="h",
                                  x=0.5, xanchor="center", y=-0.02),
                      scene=dict(aspectmode="data"), scene2=dict(aspectmode="data"))
    return fig


# 缺损覆盖最好和最差的两颗 —— 看模型在哪种洞上撑不住
_d = eval_df[eval_df["run"] == SHOW_RUN]["defect_cov_mm"].reset_index(drop=True)
show_completion(int(_d.idxmin())).show()
show_pred_vs_gt(int(_d.idxmax())).show()

## 10. 对照组

### 10.1 作者发布的预训练权重

由 [`MSN_baseline_pretrained.ipynb`](MSN_baseline_pretrained.ipynb) 产出，存在
`experiments_log/pretrained_baseline/eval_val20.csv`。**同一批 20 颗验证颅骨、同一套指标
定义、同一份已对齐的 `.npz`**，所以这是干净的对照。

⚠️ 性质要写准：这是**"通用基础模型直接应用于颅骨" vs "颅骨专精训练"**，不是同任务下
两个方法的较量。论文报的 DCD 是 1.41269，用它的权重在本项目颅骨上实测 **1.42772（差 1%）**
——说明该模型在颅骨上**并未失效**，本工作的增益来自**专精化**。这个措辞比"我们修好了它的
缺陷"准确得多，也守得住。

⚠️ 仍**不要**写 "zero-shot"：MedShapeNet 含 bones 类且部分源自 AutoImplant（SkullFix 的
来源），很可能见过颅骨。落实之前用中性表述。

### 10.2 原论文报告的数字

| | 训练数据 | CD | DCD | F1@0.05 | F1@0.03 |
|---|---|---:|---:|---:|---:|
| Table 1（全量） | 200,000 点云 / 240 类 | 0.00170 | 1.41269 | 0.9370 | 0.7408 |
| Table 2（小数据量） | 4,800 形状 | 0.002327 | 1.60467 | 0.89202 | 0.6234 |

**Table 2 那行是和本项目处境最接近的参照**。他们的训练算力是 6×A6000 六周，
本项目是单卡 40~60 分钟——**差约三个数量级**，引用绝对数字时必须交代。

⚠️ **CD 那一列不要跨项目比**：按 `calc_cd` 的线性距离定义，0.00170 会比点间距还小 35 倍，
物理上不可能——推测论文用的是 `loss.py` 里另一个基于平方距离的函数。DCD 和 F1 没有这个问题。

In [ ]:
pre = pd.read_csv(os.path.join(REPO, "experiments_log", "pretrained_baseline", "eval_val20.csv"))
best_run = eval_df.groupby("run")["defect_cov_mm"].mean().idxmin()     # 按主指标挑
mine = eval_df[eval_df["run"] == best_run]

print(f"同样 {len(pre)} 颗验证颅骨，同样的指标定义\n")
print(f"{'':<46}{'CD_t (mm)':>12}{'DCD':>10}")
print("-" * 68)
print(f"{'作者发布的预训练权重（未在颅骨上训练）':<46}"
      f"{pre['CD_t_mm'].mean():>12.3f}{pre['DCD'].mean():>10.4f}")
print(f"{'本工作（' + best_run + '，从零训练）':<46}"
      f"{mine['CD_t_mm'].mean():>12.3f}{mine['DCD'].mean():>10.4f}")
print("-" * 68)
print(f"{'改善':<46}{(1 - mine['CD_t_mm'].mean() / pre['CD_t_mm'].mean()) * 100:>11.1f}%"
      f"{(1 - mine['DCD'].mean() / pre['DCD'].mean()) * 100:>9.1f}%")
print(f"\n复现性核对：论文 Table 1 报 DCD = 1.41269，这里实测 {pre['DCD'].mean():.5f}"
      f" → 差 {abs(pre['DCD'].mean() - 1.41269) / 1.41269 * 100:.1f}%。"
      f"\n指标实现与权重加载两件事同时被验证。")